In [ ]:
!pip install bitsandbytes


In [ ]:
!pip install -U sentence-transformers bert_score nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.4 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [ ]:
import torch
import numpy as np
import re
import difflib
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from bert_score import score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm.auto import tqdm


In [ ]:
generics_kb = load_dataset('community-datasets/generics_kb')


In [ ]:
commonsense_qa = load_dataset('tau/commonsense_qa')


In [ ]:
generics_kb['train'][0]


{'source': 'Waterloo',
 'term': 'aa battery',
 'quantifier_frequency': '',
 'quantifier_number': '',
 'generic_sentence': 'AA batteries maintain the settings if the power ever goes off.',
 'score': 0.35092294216156006}

In [ ]:
commonsense_qa['train'][0]


{'id': '075e483d21c29a511267ef62bedc0461',
 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?',
 'question_concept': 'punishing',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
  'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']},
 'answerKey': 'A'}

In [ ]:
DOC_LIMIT = 5000

TOP_K_LIST = [1, 3, 5]

qa_split = commonsense_qa['validation']
NUM_QUESTIONS = len(qa_split)
subset = qa_split.select(range(NUM_QUESTIONS))

if DOC_LIMIT is None:
    documents = generics_kb['train']['generic_sentence']
else:
    documents = generics_kb['train'].select(range(DOC_LIMIT))['generic_sentence']


In [ ]:
from huggingface_hub import login
login()


In [ ]:
model_name = 'meta-llama/Llama-2-7b-hf'


In [ ]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map='auto')


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
embeddings_model = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
document_embeddings = embeddings_model.encode(documents, batch_size=64, show_progress_bar=True, convert_to_tensor=True)
document_embeddings

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

tensor([[ 0.0258,  0.0365, -0.0636,  ...,  0.0092, -0.0345, -0.0370],
        [-0.0111, -0.0863, -0.0271,  ..., -0.0550,  0.0290, -0.0216],
        [-0.0223,  0.1054, -0.0091,  ...,  0.0355, -0.0262,  0.0234],
        ...,
        [ 0.0208, -0.0022, -0.0214,  ..., -0.0083,  0.1675, -0.1291],
        [ 0.0099, -0.0314, -0.0064,  ...,  0.0116,  0.1933, -0.0804],
        [-0.0133, -0.0179, -0.0247,  ...,  0.0963,  0.1568, -0.0450]],
       device='cuda:0')

In [ ]:
results_minilm = {}
smooth = SmoothingFunction().method1

for top_k in TOP_K_LIST:
    preds = []
    refs = []
    bleu_scores = []
    for item in tqdm(subset, total=NUM_QUESTIONS):
        question = item['question']
        choices = list(zip(item['choices']['label'], item['choices']['text']))
        ref_label = item['answerKey']
        ref_text = dict(choices)[ref_label]

        if top_k > 0:
            query_embedding = embeddings_model.encode(question, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings, top_k=top_k)[0]
            context_docs = [documents[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'- {c}' for c in context_docs])
            prompt = (
                'Use the context to answer the multiple-choice question.\n\n'
                f'Context:\n{context_block}\n\n'
                f'Question: {question}\n'
                f'Choices:\n' + '\n'.join([f'{label}: {text}' for label, text in choices]) + '\n'
                'Answer:'
            )
        else:
            prompt = (
                'Answer the multiple-choice question.\n\n'
                f'Question: {question}\n'
                f'Choices:\n' + '\n'.join([f'{label}: {text}' for label, text in choices]) + '\n'
                'Answer:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Answer:' in decoded:
            raw_answer = decoded.split('Answer:')[-1].strip()
        else:
            raw_answer = decoded[len(prompt):].strip()

        raw = raw_answer.lower()
        pred_text = None
        match = re.search(r'\b([a-e])\b', raw)
        if match:
            label = match.group(1).upper()
            for lbl, text in choices:
                if lbl == label:
                    pred_text = text
        if pred_text is None:
            for _, text in choices:
                if text.lower() in raw:
                    pred_text = text
                    break
        if pred_text is None:
            pred_text = max(
                choices,
                key=lambda c: difflib.SequenceMatcher(None, raw, c[1].lower()).ratio()
            )[1]

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_minilm[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_minilm


  0%|          | 0/1221 [00:00<?, ?it/s]

In [ ]:
embeddings_model_distil = SentenceTransformer('all-distilroberta-v1')


In [ ]:
document_embeddings_distil = embeddings_model_distil.encode(documents, batch_size=64, show_progress_bar=True, convert_to_tensor=True)


In [ ]:
results_distil = {}

for top_k in TOP_K_LIST:
    preds = []
    refs = []
    bleu_scores = []
    for item in tqdm(subset, total=NUM_QUESTIONS):
        question = item['question']
        choices = list(zip(item['choices']['label'], item['choices']['text']))
        ref_label = item['answerKey']
        ref_text = dict(choices)[ref_label]

        if top_k > 0:
            query_embedding = embeddings_model_distil.encode(question, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings_distil, top_k=top_k)[0]
            context_docs = [documents[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'- {c}' for c in context_docs])
            prompt = (
                'Use the context to answer the multiple-choice question.\n\n'
                f'Context:\n{context_block}\n\n'
                f'Question: {question}\n'
                f'Choices:\n' + '\n'.join([f'{label}: {text}' for label, text in choices]) + '\n'
                'Answer:'
            )
        else:
            prompt = (
                'Answer the multiple-choice question.\n\n'
                f'Question: {question}\n'
                f'Choices:\n' + '\n'.join([f'{label}: {text}' for label, text in choices]) + '\n'
                'Answer:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Answer:' in decoded:
            raw_answer = decoded.split('Answer:')[-1].strip()
        else:
            raw_answer = decoded[len(prompt):].strip()

        raw = raw_answer.lower()
        pred_text = None
        match = re.search(r'\b([a-e])\b', raw)
        if match:
            label = match.group(1).upper()
            for lbl, text in choices:
                if lbl == label:
                    pred_text = text
        if pred_text is None:
            for _, text in choices:
                if text.lower() in raw:
                    pred_text = text
                    break
        if pred_text is None:
            pred_text = max(
                choices,
                key=lambda c: difflib.SequenceMatcher(None, raw, c[1].lower()).ratio()
            )[1]

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_distil[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_distil


In [ ]:
preds = []
refs = []
bleu_scores = []

for item in tqdm(subset, total=NUM_QUESTIONS):
    question = item['question']
    choices = list(zip(item['choices']['label'], item['choices']['text']))
    ref_label = item['answerKey']
    ref_text = dict(choices)[ref_label]

    prompt = (
        'Answer the multiple-choice question.\n\n'
        f'Question: {question}\n'
        f'Choices:\n' + '\n'.join([f'{label}: {text}' for label, text in choices]) + '\n'
        'Answer:'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'Answer:' in decoded:
        raw_answer = decoded.split('Answer:')[-1].strip()
    else:
        raw_answer = decoded[len(prompt):].strip()

    raw = raw_answer.lower()
    pred_text = None
    match = re.search(r'\b([a-e])\b', raw)
    if match:
        label = match.group(1).upper()
        for lbl, text in choices:
            if lbl == label:
                pred_text = text
    if pred_text is None:
        for _, text in choices:
            if text.lower() in raw:
                pred_text = text
                break
    if pred_text is None:
        pred_text = max(
            choices,
            key=lambda c: difflib.SequenceMatcher(None, raw, c[1].lower()).ratio()
        )[1]

    preds.append(pred_text)
    refs.append(ref_text)
    bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

_, _, f1 = score(preds, refs, lang='en', verbose=False)
zero_shot_result = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}
zero_shot_result


In [ ]:
rows = []
for top_k, metrics in results_minilm.items():
    rows.append({'setting': f'minilm_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
for top_k, metrics in results_distil.items():
    rows.append({'setting': f'distilroberta_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
rows.append({'setting': 'zero_shot', 'bleu': zero_shot_result['bleu'], 'bert_f1': zero_shot_result['bert_f1']})

rows


In [ ]:
minilm_bleu = [results_minilm[k]['bleu'] for k in TOP_K_LIST]
minilm_bert = [results_minilm[k]['bert_f1'] for k in TOP_K_LIST]
distil_bleu = [results_distil[k]['bleu'] for k in TOP_K_LIST]
distil_bert = [results_distil[k]['bert_f1'] for k in TOP_K_LIST]

print('MiniLM top-k BLEU:', minilm_bleu)
print('MiniLM top-k BERT F1:', minilm_bert)
print('DistilRoBERTa top-k BLEU:', distil_bleu)
print('DistilRoBERTa top-k BERT F1:', distil_bert)
print('Zero-shot BLEU/BERT F1:', zero_shot_result['bleu'], zero_shot_result['bert_f1'])


Task 2

In [ ]:
wikipedia_text_corpus = load_dataset('rag-datasets/rag-mini-wikipedia', 'text-corpus')
wikipedia_question_answer = load_dataset('rag-datasets/rag-mini-wikipedia', 'question-answer')


In [ ]:
wikipedia_text_corpus['passages'][0]

In [ ]:
wikipedia_question_answer['test'][0]

In [ ]:
documents_wiki = [p['passage'] for p in wikipedia_text_corpus['passages']]


In [ ]:
NUM_QUESTIONS_WIKI = len(wikipedia_question_answer['test'])
TOP_K_LIST_WIKI = [1, 3, 5]

subset_wiki = wikipedia_question_answer['test'].select(range(NUM_QUESTIONS_WIKI))

In [ ]:
embeddings_model_wiki = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
document_embeddings_wiki = embeddings_model_wiki.encode(documents_wiki, batch_size=64, show_progress_bar=True, convert_to_tensor=True)


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
results_minilm_wiki = {}
smooth = SmoothingFunction().method1

for top_k in TOP_K_LIST_WIKI:
    preds = []
    refs = []
    bleu_scores = []
    for item in tqdm(subset_wiki, total=NUM_QUESTIONS_WIKI):
        question = item['question']
        ref_text = item['answer']

        if top_k > 0:
            query_embedding = embeddings_model_wiki.encode(question, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings_wiki, top_k=top_k)[0]
            context_docs = [documents_wiki[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'Document {idx + 1}: {doc}' for idx, doc in enumerate(context_docs)])
            prompt = (
                f'Context:\n{context_block}\n\n'
                f'Answer the following question: {question}\n\n'
                'Answer:'
            )
        else:
            prompt = (
                f'Answer the following question: {question}\n\n'
                'Answer:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Answer:' in decoded:
            pred_text = decoded.split('Answer:')[-1].strip()
        else:
            pred_text = decoded[len(prompt):].strip()

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_minilm_wiki[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_minilm_wiki


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{1: {'bleu': 0.0059065327913884625, 'bert_f1': 0.812085747718811},
 3: {'bleu': 0.004577801432085154, 'bert_f1': 0.8078638315200806},
 5: {'bleu': 0.006943170603426719, 'bert_f1': 0.8123036026954651}}

In [ ]:
embeddings_model_wiki_distil = SentenceTransformer('all-distilroberta-v1')


In [ ]:
document_embeddings_wiki_distil = embeddings_model_wiki_distil.encode(documents_wiki, batch_size=64, show_progress_bar=True, convert_to_tensor=True)


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
results_distil_wiki = {}

for top_k in TOP_K_LIST_WIKI:
    preds = []
    refs = []
    bleu_scores = []
    for item in tqdm(subset_wiki, total=NUM_QUESTIONS_WIKI):
        question = item['question']
        ref_text = item['answer']

        if top_k > 0:
            query_embedding = embeddings_model_wiki_distil.encode(question, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings_wiki_distil, top_k=top_k)[0]
            context_docs = [documents_wiki[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'Document {idx + 1}: {doc}' for idx, doc in enumerate(context_docs)])
            prompt = (
                f'Context:\n{context_block}\n\n'
                f'Answer the following question: {question}\n\n'
                'Answer:'
            )
        else:
            prompt = (
                f'Answer the following question: {question}\n\n'
                'Answer:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Answer:' in decoded:
            pred_text = decoded.split('Answer:')[-1].strip()
        else:
            pred_text = decoded[len(prompt):].strip()

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_distil_wiki[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_distil_wiki


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{1: {'bleu': 0.008773993275800694, 'bert_f1': 0.8131482601165771},
 3: {'bleu': 0.009036869823702352, 'bert_f1': 0.8081874251365662},
 5: {'bleu': 0.002573163711612116, 'bert_f1': 0.8065983653068542}}

In [ ]:
preds = []
refs = []
bleu_scores = []

for item in tqdm(subset_wiki, total=NUM_QUESTIONS_WIKI):
    question = item['question']
    ref_text = item['answer']

    prompt = (
        f'Answer the following question: {question}\n\n'
        'Answer:'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'Answer:' in decoded:
        pred_text = decoded.split('Answer:')[-1].strip()
    else:
        pred_text = decoded[len(prompt):].strip()

    preds.append(pred_text)
    refs.append(ref_text)
    bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

_, _, f1 = score(preds, refs, lang='en', verbose=False)
zero_shot_result_wiki = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}
zero_shot_result_wiki


  0%|          | 0/50 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'bleu': 0.004031321935331278, 'bert_f1': 0.7861489057540894}

In [ ]:
rows_wiki = []
for top_k, metrics in results_minilm_wiki.items():
    rows_wiki.append({'setting': f'minilm_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
for top_k, metrics in results_distil_wiki.items():
    rows_wiki.append({'setting': f'distilroberta_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
rows_wiki.append({'setting': 'zero_shot', 'bleu': zero_shot_result_wiki['bleu'], 'bert_f1': zero_shot_result_wiki['bert_f1']})

rows_wiki


[{'setting': 'minilm_topk_1',
  'bleu': 0.0059065327913884625,
  'bert_f1': 0.812085747718811},
 {'setting': 'minilm_topk_3',
  'bleu': 0.004577801432085154,
  'bert_f1': 0.8078638315200806},
 {'setting': 'minilm_topk_5',
  'bleu': 0.006943170603426719,
  'bert_f1': 0.8123036026954651},
 {'setting': 'distilroberta_topk_1',
  'bleu': 0.008773993275800694,
  'bert_f1': 0.8131482601165771},
 {'setting': 'distilroberta_topk_3',
  'bleu': 0.009036869823702352,
  'bert_f1': 0.8081874251365662},
 {'setting': 'distilroberta_topk_5',
  'bleu': 0.002573163711612116,
  'bert_f1': 0.8065983653068542},
 {'setting': 'zero_shot',
  'bleu': 0.004031321935331278,
  'bert_f1': 0.7861489057540894}]

In [ ]:
inilm_bleu_wiki = [results_minilm_wiki[k]['bleu'] for k in TOP_K_LIST_WIKI]
minilm_bert_wiki = [results_minilm_wiki[k]['bert_f1'] for k in TOP_K_LIST_WIKI]
distil_bleu_wiki = [results_distil_wiki[k]['bleu'] for k in TOP_K_LIST_WIKI]
distil_bert_wiki = [results_distil_wiki[k]['bert_f1'] for k in TOP_K_LIST_WIKI]

print('MiniLM top-k BLEU:', minilm_bleu_wiki)
print('MiniLM top-k BERT F1:', minilm_bert_wiki)
print('DistilRoBERTa top-k BLEU:', distil_bleu_wiki)
print('DistilRoBERTa top-k BERT F1:', distil_bert_wiki)
print('Zero-shot BLEU/BERT F1:', zero_shot_result_wiki['bleu'], zero_shot_result_wiki['bert_f1'])


MiniLM top-k BLEU: [0.0059065327913884625, 0.004577801432085154, 0.006943170603426719]
MiniLM top-k BERT F1: [0.812085747718811, 0.8078638315200806, 0.8123036026954651]
DistilRoBERTa top-k BLEU: [0.008773993275800694, 0.009036869823702352, 0.002573163711612116]
DistilRoBERTa top-k BERT F1: [0.8131482601165771, 0.8081874251365662, 0.8065983653068542]
Zero-shot BLEU/BERT F1: 0.004031321935331278 0.7861489057540894


Task 3


In [ ]:
import pandas as pd

data = pd.read_csv('test_en_parallel.txt', sep='\t', header=None)
data = data[[0, 1]]
data.columns = ['NEGATIVE', 'POSITIVE']
data = data[1:999]
sentences_ne = data['NEGATIVE'].values.tolist()
sentences_pos = data['POSITIVE'].values.tolist()

sentences = []
for s in sentences_pos:
    sentences.append((s, 'positive'))
for s in sentences_ne:
    sentences.append((s, 'negative'))
labels = [label for _, label in sentences]
labels

sentences = [sentence for sentence, _ in sentences]


In [ ]:
NUM_SENTS_YELP = 200
TOP_K_LIST_YELP = [1, 3, 5]

if NUM_SENTS_YELP is None:
    neg_subset = sentences_ne
    pos_subset = sentences_pos
else:
    neg_subset = sentences_ne[:NUM_SENTS_YELP]
    pos_subset = sentences_pos[:NUM_SENTS_YELP]

documents_yelp = sentences_pos


In [ ]:
embeddings_model_yelp = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
document_embeddings_yelp = embeddings_model_yelp.encode(documents_yelp, batch_size=64, show_progress_bar=True, convert_to_tensor=True)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [ ]:
results_minilm_yelp = {}
smooth = SmoothingFunction().method1

for top_k in TOP_K_LIST_YELP:
    preds = []
    refs = []
    bleu_scores = []
    for neg_text, ref_text in tqdm(list(zip(neg_subset, pos_subset)), total=len(neg_subset)):
        if top_k > 0:
            query_embedding = embeddings_model_yelp.encode(neg_text, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings_yelp, top_k=top_k)[0]
            context_docs = [documents_yelp[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'Document {idx + 1}: {doc}' for idx, doc in enumerate(context_docs)])
            prompt = (
                'Use the context to rewrite the review with positive sentiment.\n\n'
                f'Context:\n{context_block}\n\n'
                f'Negative review: {neg_text}\n\n'
                'Positive rewrite:'
            )
        else:
            prompt = (
                'Rewrite the review with positive sentiment.\n\n'
                f'Negative review: {neg_text}\n\n'
                'Positive rewrite:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Positive rewrite:' in decoded:
            pred_text = decoded.split('Positive rewrite:')[-1].strip()
        else:
            pred_text = decoded[len(prompt):].strip()

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_minilm_yelp[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_minilm_yelp


  0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{1: {'bleu': 0.23341589151309988, 'bert_f1': 0.861663281917572},
 3: {'bleu': 0.11607057232241029, 'bert_f1': 0.8547313213348389},
 5: {'bleu': 0.07692559799852039, 'bert_f1': 0.8324430584907532}}

In [ ]:
embeddings_model_yelp_distil = SentenceTransformer('all-distilroberta-v1')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
document_embeddings_yelp_distil = embeddings_model_yelp_distil.encode(documents_yelp, batch_size=64, show_progress_bar=True, convert_to_tensor=True)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [ ]:
results_distil_yelp = {}

for top_k in TOP_K_LIST_YELP:
    preds = []
    refs = []
    bleu_scores = []
    for neg_text, ref_text in tqdm(list(zip(neg_subset, pos_subset)), total=len(neg_subset)):
        if top_k > 0:
            query_embedding = embeddings_model_yelp_distil.encode(neg_text, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, document_embeddings_yelp_distil, top_k=top_k)[0]
            context_docs = [documents_yelp[h['corpus_id']] for h in hits]
            context_block = '\n'.join([f'Document {idx + 1}: {doc}' for idx, doc in enumerate(context_docs)])
            prompt = (
                'Use the context to rewrite the review with positive sentiment.\n\n'
                f'Context:\n{context_block}\n\n'
                f'Negative review: {neg_text}\n\n'
                'Positive rewrite:'
            )
        else:
            prompt = (
                'Rewrite the review with positive sentiment.\n\n'
                f'Negative review: {neg_text}\n\n'
                'Positive rewrite:'
            )

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Positive rewrite:' in decoded:
            pred_text = decoded.split('Positive rewrite:')[-1].strip()
        else:
            pred_text = decoded[len(prompt):].strip()

        preds.append(pred_text)
        refs.append(ref_text)
        bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

    _, _, f1 = score(preds, refs, lang='en', verbose=False)
    results_distil_yelp[top_k] = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}

results_distil_yelp


  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{1: {'bleu': 0.23746430844633729, 'bert_f1': 0.8755310773849487},
 3: {'bleu': 0.10591277502012708, 'bert_f1': 0.8268740773200989},
 5: {'bleu': 0.08498298186977166, 'bert_f1': 0.8248663544654846}}

In [ ]:
preds = []
refs = []
bleu_scores = []

for neg_text, ref_text in tqdm(list(zip(neg_subset, pos_subset)), total=len(neg_subset)):
    prompt = (
        'Rewrite the review with positive sentiment.\n\n'
        f'Negative review: {neg_text}\n\n'
        'Positive rewrite:'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'Positive rewrite:' in decoded:
        pred_text = decoded.split('Positive rewrite:')[-1].strip()
    else:
        pred_text = decoded[len(prompt):].strip()

    preds.append(pred_text)
    refs.append(ref_text)
    bleu_scores.append(sentence_bleu([ref_text.split()], pred_text.split(), smoothing_function=smooth))

_, _, f1 = score(preds, refs, lang='en', verbose=False)
zero_shot_result_yelp = {'bleu': float(np.mean(bleu_scores)), 'bert_f1': float(f1.mean())}
zero_shot_result_yelp


  0%|          | 0/200 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'bleu': 0.09704747488751365, 'bert_f1': 0.8433199524879456}

In [ ]:
rows_yelp = []
for top_k, metrics in results_minilm_yelp.items():
    rows_yelp.append({'setting': f'minilm_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
for top_k, metrics in results_distil_yelp.items():
    rows_yelp.append({'setting': f'distilroberta_topk_{top_k}', 'bleu': metrics['bleu'], 'bert_f1': metrics['bert_f1']})
rows_yelp.append({'setting': 'zero_shot', 'bleu': zero_shot_result_yelp['bleu'], 'bert_f1': zero_shot_result_yelp['bert_f1']})

rows_yelp


[{'setting': 'minilm_topk_1',
  'bleu': 0.23341589151309988,
  'bert_f1': 0.861663281917572},
 {'setting': 'minilm_topk_3',
  'bleu': 0.11607057232241029,
  'bert_f1': 0.8547313213348389},
 {'setting': 'minilm_topk_5',
  'bleu': 0.07692559799852039,
  'bert_f1': 0.8324430584907532},
 {'setting': 'distilroberta_topk_1',
  'bleu': 0.23746430844633729,
  'bert_f1': 0.8755310773849487},
 {'setting': 'distilroberta_topk_3',
  'bleu': 0.10591277502012708,
  'bert_f1': 0.8268740773200989},
 {'setting': 'distilroberta_topk_5',
  'bleu': 0.08498298186977166,
  'bert_f1': 0.8248663544654846},
 {'setting': 'zero_shot',
  'bleu': 0.09704747488751365,
  'bert_f1': 0.8433199524879456}]

In [ ]:
minilm_bleu_yelp = [results_minilm_yelp[k]['bleu'] for k in TOP_K_LIST_YELP]
minilm_bert_yelp = [results_minilm_yelp[k]['bert_f1'] for k in TOP_K_LIST_YELP]
distil_bleu_yelp = [results_distil_yelp[k]['bleu'] for k in TOP_K_LIST_YELP]
distil_bert_yelp = [results_distil_yelp[k]['bert_f1'] for k in TOP_K_LIST_YELP]

print('MiniLM top-k BLEU:', minilm_bleu_yelp)
print('MiniLM top-k BERT F1:', minilm_bert_yelp)
print('DistilRoBERTa top-k BLEU:', distil_bleu_yelp)
print('DistilRoBERTa top-k BERT F1:', distil_bert_yelp)
print('Zero-shot BLEU/BERT F1:', zero_shot_result_yelp['bleu'], zero_shot_result_yelp['bert_f1'])


MiniLM top-k BLEU: [0.23341589151309988, 0.11607057232241029, 0.07692559799852039]
MiniLM top-k BERT F1: [0.861663281917572, 0.8547313213348389, 0.8324430584907532]
DistilRoBERTa top-k BLEU: [0.23746430844633729, 0.10591277502012708, 0.08498298186977166]
DistilRoBERTa top-k BERT F1: [0.8755310773849487, 0.8268740773200989, 0.8248663544654846]
Zero-shot BLEU/BERT F1: 0.09704747488751365 0.8433199524879456


In [ ]:
best_bleu_setting = max(rows_yelp, key=lambda r: r['bleu'])
best_bert_setting = max(rows_yelp, key=lambda r: r['bert_f1'])

print('Best BLEU setting:', best_bleu_setting['setting'], best_bleu_setting['bleu'])
print('Best BERT F1 setting:', best_bert_setting['setting'], best_bert_setting['bert_f1'])
print('Compare these scores to earlier exercises to judge improvement.')


Best BLEU setting: distilroberta_topk_1 0.23746430844633729
Best BERT F1 setting: distilroberta_topk_1 0.8755310773849487
Compare these scores to earlier exercises to judge improvement.
